In [1]:
import pandas as pd


def periodicSave(moderatorLogs, batch_id):
    try:
        df = pd.DataFrame(moderatorLogs)
        df.to_json(
            f"./moderatorLogs_{batch_id}.json",
            orient="records"
        )
        print(f"💾 Saved batch {batch_id} ({len(df)} actions)")
        return True
    except Exception as e:
        print(f"❌ Error saving logs: {e}")
        return False

In [2]:
import urllib.request
import json
import time
from bs4 import BeautifulSoup


def extract_moderator_logs(logsLink, retries=5, delay=10):
    """
    Extract moderator actions from a Wayback snapshot.
    Returns a list of moderation actions (dicts).
    """

    headers = {'User-Agent': 'Mozilla/5.0'}

    for attempt in range(retries):
        try:
            print(f"Fetching: {logsLink} (Attempt {attempt+1}/{retries})")

            # ---- Step 1: fetch HTML page ----
            req = urllib.request.Request(logsLink, headers=headers)
            with urllib.request.urlopen(req) as response:
                html = response.read()

            soup = BeautifulSoup(html, "html.parser")

            # ---- Step 2: extract snapshot JSON link ----
            iframe = soup.find("iframe", id="playback")
            if not iframe or not iframe.get("src"):
                print("⚠️ No playback iframe found")
                return []

            snapshot_link = iframe["src"]

            # ---- Step 3: fetch JSON snapshot ----
            req = urllib.request.Request(snapshot_link, headers=headers)
            with urllib.request.urlopen(req) as response:
                raw_json = response.read().decode("utf-8")

            json_data = json.loads(raw_json)

            # ---- Step 4: extract moderation actions ----
            actions = json_data.get("data", {}).get("children", [])

            print(f"✅ Extracted {len(actions)} moderation actions")
            return actions

        except Exception as e:
            print(f"⚠️ Error on attempt {attempt+1}/{retries}: {e}")
            if attempt < retries - 1:
                print(f"⏳ Retrying in {delay}s...")
                time.sleep(delay)

    print(f"❌ All attempts failed for {logsLink}")
    return []


In [ ]:
import pandas as pd


logsSnapshots = pd.read_csv("moderatorLogs.csv")

moderatorLogs = []
batch_id = 0

for index in range(2185 ,len(logsSnapshots)):
    print(f"🔎 Working with Trace {index+1} / {len(logsSnapshots)}")

    actions = extract_moderator_logs(logsSnapshots.loc[index, "url href"])

    # actions is a LIST of moderation actions
    if actions:
        moderatorLogs.extend(actions)

    # Save every 10 traces (not at index 0)
    if (index + 1) % 10 == 0:
        periodicSave(moderatorLogs, batch_id)
        moderatorLogs = []
        batch_id += 1

# ---- Save remaining logs ----
if moderatorLogs:
    periodicSave(moderatorLogs, batch_id)

# ---- Save snapshot metadata ----
logsSnapshots.to_json(
    "moderatorLogs.json",
    orient="records"
)


🔎 Working with Trace 1 / 9999
Fetching: https://web.archive.org/web/20210104155841/http://logs.mod.rcoronavirus.org/.json (Attempt 1/5)
✅ Extracted 101 moderation actions
🔎 Working with Trace 2 / 9999
Fetching: https://web.archive.org/web/20200502181943/http://logs.mod.rcoronavirus.org/.json?after=ModAction_00023c00-8672-11ea-bd7d-0e2cd15b49ff (Attempt 1/5)
✅ Extracted 101 moderation actions
🔎 Working with Trace 3 / 9999
Fetching: https://web.archive.org/web/20200501184654/http://logs.mod.rcoronavirus.org/.json?after=ModAction_0002e60e-8b4b-11ea-b27a-129b20e9ebf2 (Attempt 1/5)
✅ Extracted 101 moderation actions
🔎 Working with Trace 4 / 9999
Fetching: https://web.archive.org/web/20200423064444/http://logs.mod.rcoronavirus.org/.json?after=ModAction_0006d3dc-7623-11ea-a6b9-0e12bb0e6841 (Attempt 1/5)
✅ Extracted 101 moderation actions
🔎 Working with Trace 5 / 9999
Fetching: https://web.archive.org/web/20200513153009/http://logs.mod.rcoronavirus.org/.json?after=ModAction_00075840-9511-11ea-

In [1]:
import os
import json
from glob import glob

# Folder containing  JSON files
folder_path = "logsData"
output_file = "logsDataset.json"

# Get all JSON files in the folder
json_files = glob(os.path.join(folder_path, "*.json"))

merged_data = []

# Read and merge all JSON files
for file in json_files:
    with open(file, "r", encoding="utf-8") as f:
        data = json.load(f)
        # If each JSON file is a list of entries
        if isinstance(data, list):
            merged_data.extend(data)
        else:
            merged_data.append(data)

# Save merged data to a new file
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(merged_data, f, indent=4)

# Delete original files
for file in json_files:
    os.remove(file)

print(f"✅ Merged {len(json_files)} files into '{output_file}' and deleted originals.")


✅ Merged 218 files into 'logsDataset.json' and deleted originals.
